# Spaceship Titanic — Final Submission

Final model from the experiment log (see `../README.md` and `../EXPERIMENTS.md`): a weighted
soft-voting ensemble of tuned CatBoost and LightGBM (2:1), trained on the **full** training
set (no held-out split — CV already validated the approach, so we use 100% of available
labels for the final fit). Self-contained: hardcodes the finalized feature set and
hyperparameters rather than depending on objects from `02_baseline_modeling.ipynb`.

Expected accuracy based on 5-fold CV: **0.8087**.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from src.features import engineer_features, SPEND_COLS

raw_train = pd.read_csv("../data/train.csv")
raw_test = pd.read_csv("../data/test.csv")
train, test = engineer_features(raw_train, raw_test)

print("train:", train.shape, " test:", test.shape)

train: (8693, 22)  test: (4277, 21)


In [2]:
# Finalized feature set (see EXPERIMENTS.md for the ablation work that produced this)
FINAL_CATEGORICAL = ["HomePlanet", "Destination", "Side", "GroupSizeBin"]
FINAL_BOOLEAN = ["CryoSleep", "VIP", "IsSolo"]
FINAL_NUMERIC = ["Age", "CabinNum"] + SPEND_COLS
FINAL_COLS = FINAL_CATEGORICAL + FINAL_NUMERIC + FINAL_BOOLEAN

X = train[FINAL_COLS]
y = train["Transported"].astype(int)
X_test = test[FINAL_COLS]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), FINAL_CATEGORICAL),
    ("num", "passthrough", FINAL_NUMERIC),
    ("bool", "passthrough", FINAL_BOOLEAN),
])

# Hardcoded best hyperparameters from grid search (EXPERIMENTS.md)
final_model = VotingClassifier(
    estimators=[
        ("catboost", Pipeline([
            ("preprocess", preprocessor),
            ("model", CatBoostClassifier(depth=4, iterations=400, learning_rate=0.05,
                                          random_state=42, verbose=False)),
        ])),
        ("lgbm", Pipeline([
            ("preprocess", preprocessor),
            ("model", LGBMClassifier(num_leaves=10, n_estimators=200, learning_rate=0.05,
                                      random_state=42, verbose=-1)),
        ])),
    ],
    voting="soft",
    weights=[2, 1],
)

# Sanity check: re-confirm CV accuracy matches the experiment log before trusting this model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sanity_scores = cross_val_score(final_model, X, y, cv=cv, scoring="accuracy")
print(f"Sanity-check CV accuracy: {sanity_scores.mean():.4f} (std={sanity_scores.std():.4f})")
print("Expected from EXPERIMENTS.md: 0.8087")

/Users/micahb/kaggle_competitions/spaceship-titanic/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/micahb/kaggle_competitions/spaceship-titanic/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/micahb/kaggle_competitions/spaceship-titanic/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/micahb/kaggle_competitions/spaceship-titanic/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Sanity-check CV accuracy: 0.8087 (std=0.0049)
Expected from EXPERIMENTS.md: 0.8087


/Users/micahb/kaggle_competitions/spaceship-titanic/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Fit on full training data and predict on test set

In [3]:
final_model.fit(X, y)
test_preds = final_model.predict(X_test).astype(bool)

print("Predicted Transported proportion:", test_preds.mean())
print("Training set actual proportion:  ", y.mean())

Predicted Transported proportion: 0.5316810848725743
Training set actual proportion:   0.5036236051995858


/Users/micahb/kaggle_competitions/spaceship-titanic/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [4]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_preds,
})

sample_submission = pd.read_csv("../data/sample_submission.csv")
assert (submission["PassengerId"].values == sample_submission["PassengerId"].values).all(), \
    "PassengerId order doesn't match sample_submission.csv"
assert submission.shape == sample_submission.shape

submission_path = "../submissions/submission_v1_catboost_lgbm_ensemble.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()

Saved: ../submissions/submission_v1_catboost_lgbm_ensemble.csv


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
